In [1]:
import pandas as pd
import os

# 1. Define the files you want to merge
csv_files = [
    "aes.csv", 
    "picorv32.csv", 
    "sha256.csv", 
    "ethmac.csv", 
    "ethmac2.csv", 
    "experiment_log.csv"
]

dataframes = []

print("Aggregating CSV files...")

# 2. Loop through and load each file
for file in csv_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        dataframes.append(df)
        print(f" -> Loaded {file} ({len(df)} rows)")
    else:
        print(f" -> WARNING: {file} not found. Skipping.")

# 3. Stack them all together
if dataframes:
    unified_df = pd.concat(dataframes, ignore_index=True)
    
    # 4. OVERWRITE the local run_id with a continuous GLOBAL run_id
    unified_df['run_id'] = range(len(unified_df))
    
    # 5. Save the unified manifest
    output_filename = "unified_manifest.csv"
    unified_df.to_csv(output_filename, index=False)
    
    print("-" * 40)
    print(f"SUCCESS: Created {output_filename} with {len(unified_df)} total rows.")
    print(f"Global run_id spans from 0 to {len(unified_df) - 1}.")
else:
    print("No dataframes were loaded. Check your file paths.")

Aggregating CSV files...
 -> Loaded aes.csv (1040 rows)
 -> Loaded picorv32.csv (1220 rows)
 -> Loaded sha256.csv (1230 rows)
 -> Loaded ethmac.csv (270 rows)
 -> Loaded ethmac2.csv (540 rows)
 -> Loaded experiment_log.csv (1100 rows)
----------------------------------------
SUCCESS: Created unified_manifest.csv with 5400 total rows.
Global run_id spans from 0 to 5399.


In [2]:
import pandas as pd
import torch

# 1. Load your newly aggregated unified manifest
input_filename = "unified_manifest.csv"
output_filename = "unified_manifest_normalized.csv"

print(f"Loading {input_filename}...")
df = pd.read_csv(input_filename)

# 2. Define the columns that need to be globally scaled
cols_to_scale = [
    'cts_max_wire', 'cts_buf_dist', 'cts_cluster_size', 'cts_cluster_dia', 
    'skew_setup', 'power_total', 'wirelength'
]

scaler_stats = {}

print("Calculating Global Z-Scores...")
# 3. Calculate stats and create new appended columns
for col in cols_to_scale:
    g_mean = df[col].mean()
    g_std = df[col].std()
    
    # Save to dictionary for inference later
    scaler_stats[col] = {'mean': g_mean, 'std': g_std}
    
    # Create the new column with a 'z_' prefix
    df[f'z_{col}'] = (df[col] - g_mean) / (g_std + 1e-6)
    
    print(f" -> Processed {col}: Mean={g_mean:.4f}, Std={g_std:.4f}")

# 4. Save the new CSV with appended columns
df.to_csv(output_filename, index=False)
print(f"\nSUCCESS: Saved normalized dataset to {output_filename}")

# 5. Save the global scaler dictionary for inference!
torch.save(scaler_stats, 'global_scaler.pt')
print("SUCCESS: Saved global_scaler.pt for production inference.")

Loading unified_manifest.csv...
Calculating Global Z-Scores...
 -> Processed cts_max_wire: Mean=204.1924, Std=43.4221
 -> Processed cts_buf_dist: Mean=110.0870, Std=23.2434
 -> Processed cts_cluster_size: Mean=21.0647, Std=5.4257
 -> Processed cts_cluster_dia: Mean=52.6399, Std=10.3139
 -> Processed skew_setup: Mean=0.7319, Std=0.1557
 -> Processed power_total: Mean=0.0479, Std=0.0246
 -> Processed wirelength: Mean=742573.2876, Std=428639.7985

SUCCESS: Saved normalized dataset to unified_manifest_normalized.csv
SUCCESS: Saved global_scaler.pt for production inference.
